# HealthGuard — Triage Risk Engine: perfect training notebook

**EVOLVEX IT SOLUTIONS PVT. LTD.** · Part M deliverable (runs on Kaggle or any Jupyter).

This notebook collects datasets, runs the open model bake-off (XGBoost · LightGBM ·
ExtraTrees · RandomForest · sklearn MLP · TensorFlow Keras MLP · soft blends),
applies the deterministic red-flag + safety-floor layer, picks the best model under
safety constraints, and distils the winner to the browser scorer with a float-exact
parity probe.

**Triage = workflow urgency (T0–T4), never diagnosis. AI assistive only.**

### Kaggle API note
To pull Kaggle datasets, place a full API token at `~/.kaggle/kaggle.json`
(`{"username": "…", "key": "…"}` — both fields) or set `KAGGLE_USERNAME` /
KAGGLE_KEY. The token uploaded during this project contained **only a username**,
so Kaggle API calls cannot authenticate; everything below therefore trains on the
protocol-labelled generator + the UCI/OpenML real cohorts (collected in `data/`).

In [ ]:
# setup (Kaggle: run once) --------------------------------------------
!pip install -q xgboost lightgbm scikit-learn joblib
!pip install -q tensorflow-cpu  # optional neuro candidate
import os, json, gc
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

## 1 — Data collection

**Why a protocol-labelled generator is the label source of truth:** public ED
triage-acuity datasets (MIMIC-IV-ED, PhysioNet Challenge) are credentialed-gated.
Real clinical cohorts below (UCI/OpenML) have NO triage labels — they serve as
realism probes and specialist pretraining data, while `triage_synth.generate`
produces 200,000 protocol-consistent rows (vitals + 16 symptoms + 8 comorbidities
+ derived scores) labelled T0–T4 by the consensus ED protocol rules.

In [ ]:
# real cohorts (UCI direct CSVs — already fetched to data/; re-fetch if needed)
REAL = {
    'ilpd':        'https://archive.ics.uci.edu/static/public/45/data.csv',
    'diabetes571': 'https://archive.ics.uci.edu/static/public/571/data.csv',
    'heartfail':   'https://archive.ics.uci.edu/static/public/519/data.csv',
    'spectf':      'https://archive.ics.uci.edu/static/public/529/data.csv',
    'maternal':    'https://archive.ics.uci.edu/static/public/863/data.csv',
}
os.makedirs('data', exist_ok=True)
for k, url in REAL.items():
    p = f'data/{k}.csv'
    if not os.path.exists(p):
        !curl -sfL {url} -o {p}
    print(k, os.path.getsize(p), 'bytes')

# protocol-labelled training pool (the local repo has the same generator)
from triage_synth import generate
X, y, meta = generate(n=200_000)
X = X.drop(columns=['p_dm','p_cad','p_stroke','p_sepsis','p_liver','p_maternal'])
feats = list(X.columns)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)
X_tr.shape, X_te.shape

## 2 — Safety layer (identical for every candidate)

Deterministic red-flag rules (11 CRITICAL codes → T0) can only escalate;
probability floors cap under-triage (`P(T0∪T1) ≥ .30 → cap T1`, `≥ .14 → cap T2`,
`P(T0∪T1∪T2) ≥ .55 → cap T2`); disagreement/unclear → `requires_human_review`.
Selection is **macro-F1 subject to** `T0-deep-to-T2+ ≤ 0.004` and
`beyond-adjacent ≤ 0.015` — accuracy never buys back under-triage.

In [ ]:
from train_triage import safe_predict, rule_scan, CRITICAL_RULES

CLASSES = [0, 1, 2, 3, 4]
def evaluate(name, proba):
    y_pred, review = safe_predict(np.asarray(proba), CLASSES, X_te)
    rep = classification_report(y_te, y_pred, labels=CLASSES, output_dict=True, zero_division=0)
    y = np.asarray(y_te); m01 = np.isin(y, [0, 1])
    m = dict(
        macro_f1=round(float(rep['macro avg']['f1-score']), 4),
        T0_deep=round(float(np.mean(y_pred[y == 0] > 1)), 4),
        beyond=round(float(np.mean(y_pred[m01] > np.maximum(y[m01], 1))), 4),
        over=round(float(np.mean(y_pred[y >= 3] <= 1)), 4),
    )
    m['feasible'] = m['T0_deep'] <= 0.004 and m['beyond'] <= 0.015
    print(name, m)
    return m

## 3 — Bake-off: fit → score → free (memory ladder)

Tabular data: boosted trees are the prior favourite (and they won — see results),
but the client asked for *what is best*, so tensorflow neuro and classic forests
compete on the exact same holdout.

In [ ]:
results, probas = {}, {}

from xgboost import XGBClassifier
xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.06, subsample=0.9,
                    colsample_bytree=0.9, min_child_weight=2, reg_lambda=1.5,
                    objective='multi:softprob', num_class=5, tree_method='hist',
                    random_state=42, n_jobs=2, verbosity=0).fit(X_tr, y_tr)
probas['xgboost'] = xgb.predict_proba(X_te); joblib.dump(xgb, 'models/web_xgb.joblib')
del xgb; gc.collect()

import lightgbm as lgb
lgbm = lgb.LGBMClassifier(n_estimators=400, num_leaves=31, learning_rate=0.05, subsample=0.9,
                          colsample_bytree=0.9, min_child_weight=5, reg_lambda=1.5,
                          random_state=42, n_jobs=2, verbose=-1).fit(X_tr, y_tr)
probas['lightgbm'] = lgbm.predict_proba(X_te); joblib.dump(lgbm, 'models/web_lgbm.joblib')
del lgbm; gc.collect()

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
et = ExtraTreesClassifier(n_estimators=200, min_samples_leaf=4, n_jobs=2, random_state=42).fit(X_tr, y_tr)
probas['extratrees'] = et.predict_proba(X_te); del et; gc.collect()
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=4, n_jobs=2, random_state=42).fit(X_tr, y_tr)
probas['randomforest'] = rf.predict_proba(X_te); del rf; gc.collect()

from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(128, 64, 32), early_stopping=True, max_iter=60,
                    random_state=42).fit(X_tr, y_tr)
probas['mlp_sklearn'] = mlp.predict_proba(X_te); del mlp; gc.collect()

import tensorflow as tf  # neuro candidate
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(X_tr)
nn = tf.keras.Sequential([
    tf.keras.layers.Input((len(feats),)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(5, activation='softmax')])
nn.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
nn.fit(sc.transform(X_tr), np.asarray(y_tr), epochs=16, batch_size=512, verbose=0)
probas['mlp_tensorflow'] = nn.predict(sc.transform(X_te), verbose=0)
del nn, sc; tf.keras.backend.clear_session(); gc.collect()

probas['blend_xgb_lgbm'] = 0.5 * probas['xgboost'] + 0.5 * probas['lightgbm']
probas['blend_xgb_lgbm_et'] = (probas['xgboost'] + probas['lightgbm'] + probas['extratrees']) / 3

for name, p in probas.items():
    results[name] = evaluate(name, p)
best = max((n for n in results if results[n]['feasible']), key=lambda n: results[n]['macro_f1'])
print('WINNER:', best)

## 4 — Result (this project's run, 2026-09-25)

| model | macro-F1 | T0-deep | beyond-adj | over | feasible |
|---|---|---|---|---|---|
| xgboost | 0.9124 | 0.0018 | 0.0135 | 0.0012 | ✅ |
| lightgbm | 0.9127 | 0.0014 | 0.0136 | 0.0019 | ✅ |
| extratrees | 0.8457 | 0.0018 | 0.0121 | 0.0360 | ✅ |
| randomforest | 0.9061 | 0.0009 | 0.0131 | 0.0026 | ✅ |
| mlp_sklearn (neuro) | 0.9000 | 0.0041 | 0.0144 | 0.0021 | ❌ safety |
| mlp_tensorflow (neuro) | 0.9012 | 0.0023 | 0.0139 | 0.0030 | ✅ |
| **blend_xgb_lgbm** | **0.9129** | **0.0014** | 0.0134 | 0.0016 | ✅ **winner** |
| blend_xgb_lgbm_et | 0.9072 | 0.0009 | 0.0130 | 0.0014 | ✅ |

**Answer to "is it xgboost or any other model":** the deployed engine is the
**XGBoost + LightGBM soft blend** (probability-space 50/50), chosen by open
bake-off — boosted trees beat both forests and both neural nets on this
45-feature tabular task, and the blend edges out either booster alone while
keeping the safety profile. The full 4-model clinical stack (with specialist
risk heads + MoE + group vote, macro-F1 0.9388) remains the server-side engine.

## 5 — Distil to the browser + parity probe

Trees are exported to flat index-addressed arrays; a per-class base margin makes
the JS scorer reproduce `predict_proba` exactly. XGBoost trees compare
`f32(x) < f32(thresh)`; LightGBM trees compare `x <= thresh` in float64 — each
exported tree carries its comparator (`c`) and float mode (`ff`). Parity on this
run: **1.5e-05 max residual**.

In [ ]:
# run the project exporter (parses xgb/lgbm dumps, calibrates base, writes
# public/js/triage-model-data.js + reports/web_model_metrics.json)
%run src/train_web_best.py

## 6 — What ships

- `triage-model-data.js` — bake-off winner, consumed by `public/js/triage-engine.js`
  (identical rules + safety layer in-browser).
- Honest web metrics (20k holdout, safety layer included): **macro-F1 0.9129**,
  recalls T0 0.893 / T1 0.954 / T2 0.878 / T3 0.947 / T4 0.878,
  T0-deep-to-T2+ **0.0014**, beyond-adjacent 0.0134, over-triage 0.0016.
- Triage class is **workflow urgency only — never diagnosis**.